# Notebook 01 — Data Pipeline (2025 BDB)

**Dataset:** 2025 NFL Big Data Bowl — 2022 season, weeks 1–9  
**Goal:** Load, filter, standardize, and clip all tracking + reference data into clean outputs for downstream notebooks.

**Key schema differences from 2023 BDB:**
- Tracking files: `tracking_week_1.csv` – `tracking_week_9.csv` (underscore format)
- `pffScoutingData.csv` → replaced by `player_play.csv`
- Pass rush filter: `wasInitialPassRusher == True` (was `pff_role == 'Pass Rush'`)
- Dropback filter: `isDropback == True` (was `passResult.isin([...])`)
- Tracking column: `club` (was `team`)
- New tracking column: `frameType`

**Outputs:**
- `outputs/tracking_clipped.parquet` + `outputs/tracking_clipped.csv`
- `outputs/edge_player_play.csv`

## Section 1 — Imports & Paths

## Section 2 — Load Tracking Data (Weeks 1–9)

Files are named `tracking_week_1.csv` through `tracking_week_9.csv`.

## Section 3 — Load Reference Tables

Four files: `games.csv`, `plays.csv`, `players.csv`, `player_play.csv`

## Section 4 — Filter to Dropback Plays

Use `isDropback == True` from `plays` (replaces the `passResult` filter from 2023 BDB).

## Section 5 — Filter player_play to EDGE Pass Rushers

Two conditions combined:
1. `wasInitialPassRusher == True` from `player_play`
2. `position IN ('DE', 'OLB')` from `players`

Then cross-filter with dropback plays.

**Note:** This is a role-based filter, not an alignment-based filter. `wasInitialPassRusher` identifies players assigned a pass rush role — it does not guarantee snap alignment at the EDGE. The 2025 BDB removed `pff_positionLinedUp`. Verify by printing `position.value_counts()` — expect overwhelmingly DE and OLB.

## Section 6 — Coordinate Standardization

Same logic as before: flip x when `playDirection == 'left'` so all plays have offense attacking right (increasing x_std).  
`playDirection` is in the tracking DataFrame. Tracking column for team is now `club` (not `team`).

## Section 7 — Build Snap Frame Lookup & Clip Tracking

Same logic as before:
1. Filter tracking to dropback plays (inner merge with `dropback_plays[['gameId','playId']]`)
2. Build snap_frame lookup from `event == 'ball_snap'`
3. Merge snap_frame onto tracking
4. Compute `frame_offset = frameId - snap_frame`
5. Clip to `0 <= frame_offset <= 25`

## Section 8 — Save Outputs & Final Verification